<div style="background:linear-gradient(135deg,#0a2540 0%,#1a3a5c 60%,#0f3460 100%);
            padding:40px 30px;border-radius:12px;text-align:center;margin-bottom:10px;">
  <h1 style="color:#f4a261;font-size:2em;margin:0 0 8px;">
    🔬 Ciencia de Datos en Descubrimiento de Fármacos
  </h1>
  <h2 style="color:#a8dadc;font-size:1.2em;font-weight:400;margin:0 0 16px;">
    12 · Graph neural networks
  </h2>
  <p style="color:#cdd6f4;font-size:0.95em;max-width:640px;margin:0 auto;line-height:1.6;">
    Universidad Nacional de Colombia · Extensión UNAL 2026<br>
    <em>Semana 6 — Del dato curado al modelo predictivo</em>
  </p>
</div>

# Redes Neuronales de Grafos (Graph Neural Networks)

---
### En esta lección aprenderás:

* cómo las moléculas pueden representarse como grafos.
* cómo funcionan las operaciones básicas de convolución en grafos.
* cómo implementar una GNN simple usando clases de PyTorch.
* las ventajas de las GNNs sobre los fingerprints.


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils import data
import math
from sklearn.metrics import roc_auc_score
import sys
from os.path import exists
if 'google.colab' in sys.modules:
    !pip install rdkit==2022.3.4
    if exists("utils.py") == False:
        !wget https://raw.githubusercontent.com/kochgroup/intro_pharma_ai/main/utils/onehotencoder.py
    %run onehotencoder.py
else:
    %run ../utils/onehotencoder.py
from rdkit import Chem
from rdkit.Chem.rdmolops import GetAdjacencyMatrix

np.set_printoptions(linewidth=300)

In [ ]:
# Laden der Daten
data_tox = pd.read_csv("https://raw.githubusercontent.com/filipsPL/tox21_dataset/master/compounds/sr-mmp.tab", sep = "\t")
data_tox = data_tox.iloc[:,1:] # all columns except the first one (index 0) are selected
data_tox.columns = ["smiles", "activity"]
data_tox.head()

## Matriz de Adyacencia y Matriz de Features One-Hot

Desafortunadamente, no podemos usar directamente las moléculas representadas como SMILES en una red neuronal. 
Primero debemos convertirlas a una representación matemática.

Una molécula puede representarse como un **grafo**:
- Los **nodos** son los átomos
- Las **aristas** son los enlaces químicos

Este grafo puede representarse mediante:
1. Una **matriz de adyacencia** $A$: matriz cuadrada donde $A_{ij}=1$ si existe un enlace entre los átomos $i$ y $j$
2. Una **matriz de features** $X$: contiene las propiedades de cada átomo (tipo de átomo, carga, etc.)

Para el one-hot encoding de los átomos usamos la función `onehotencode`.

In [ ]:
mols = [Chem.MolFromSmiles(x) for x in data_tox['smiles']]
A = [GetAdjacencyMatrix(x) for x in mols]
print(A[1])

Como features de los átomos usamos únicamente el tipo de átomo. También lo codificaremos con one-hot encoding.

Para el one-hot encoding de los tipos de átomos usamos la función `onehotencode`. 
Aplícala a la lista de moléculas `mols`.

**¿Puedes completar el código?**

In [ ]:
feat = onehotencode(_____)
feat[1]

<details>
<summary><strong>Solución:</strong></summary>

```python
feat = onehotencode(mols)
feat[1]
```
</details>

Arriba puedes ver cómo se ve una matriz de features para una molécula. 
Si miramos el `.shape`, podemos ver que tiene 24 filas (24 átomos) y 25 columnas (25 tipos de átomos diferentes). 
Cada fila corresponde a un átomo y cada columna a un tipo de átomo diferente. 
Si el átomo es del tipo correspondiente, el valor en esa posición es 1; de lo contrario, es 0.

In [ ]:
feat[1].shape

Es posible que hayas notado que aún hay ceros en la diagonal de la matriz de adyacencia. 
En una convolución de grafos, sin embargo, también queremos incluir la información del propio átomo — no solo la de sus vecinos. 
Para ello, ponemos la diagonal a 1 (autoconexiones):

In [ ]:
for matrix in A:
    np.fill_diagonal(matrix, 1)
print(A[1])

## Convolución de Grafos

Ahora queremos pasar la información de cada nodo a los nodos vecinos a lo largo de las aristas. 
Esto se llama **convolución de grafos**. La operación básica es:

$$H^{(l+1)} = \hat{D}^{-1} \hat{A} H^{(l)} W^{(l)}$$

Donde:
- $\hat{A}$ es la matriz de adyacencia con autoconexiones
- $\hat{D}$ es la matriz de grado diagonal (suma de conexiones por nodo)
- $H^{(l)}$ es la matriz de features en la capa $l$
- $W^{(l)}$ son los pesos aprendibles de la capa $l$

Primero necesitamos la matriz de grado $\hat{D}$:

In [ ]:
D =[]
for matrix in A:
    D.append(np.diag(np.sum(matrix, axis=1)))
print(D[1])

Pero no necesitamos $\hat{D}$. Necesitamos la inversa de esta matriz. 
Sin $\hat{D}^{-1}$, $\hat{A}X$ sumaría los features de los vecinos. 
Con $\hat{D}^{-1}$, en cambio, los **promedia**. 
Esto es importante para que los nodos con muchos vecinos no dominen el aprendizaje:

In [ ]:
DA = []
for i in range(len(D)):
    DA.append(np.matmul(np.linalg.inv(D[i]),A[i]))
DA[0]

Ahora tenemos la lista de matrices de adyacencia `DA`, que contiene la información sobre la estructura de la molécula. 
La lista `feat` contiene la información sobre los átomos. 
Convertimos ambas a tensores de PyTorch:

In [ ]:
DA = [torch.tensor(x,dtype=torch.float32) for x in DA] 
feat = [torch.tensor(x,dtype=torch.float32) for x in feat] 
labels = [torch.tensor([x], dtype=torch.float32) for x in data_tox['activity']]

## Capa de Convolución de Grafos

Queremos aprovechar PyTorch para la convolución de grafos. 
Al igual que la semana pasada, usaremos clases de PyTorch para definir la capa.

La capa de convolución de grafos realiza la siguiente operación:

$$H^{(l+1)} = \text{ReLU}(\hat{D}^{-1} \hat{A} H^{(l)} W^{(l)} + b^{(l)})$$

La multiplicación $\hat{D}^{-1} \hat{A} H^{(l)}$ agrega la información de los vecinos. 
Luego, $W^{(l)}$ realiza la transformación lineal aprendible.

**¿Puedes completar la clase `GraphConvolution`?**

In [ ]:
class GraphConvolution(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        self.bias = nn.Parameter(torch.FloatTensor(out_features))
        self.reset_parameters()
    
    def reset_parameters(self):
        stdv = 1. / math.sqrt(self.weight.size(1))
        self.weight.data.uniform_(0, stdv)
        self.bias.data.uniform_(-stdv, stdv)
        
    def forward(self, x, adj):
        support = torch.mm(x, self.weight)
        output = torch.mm(adj, support)
        return output + self.bias
    
    def __repr__(self):
        return self.__class__.__name__ + ' (' \
               + 'in_features=' + str(self.in_features) + ', ' \
               + 'out_features=' + str(self.out_features) + ')'

También podemos verificar si nuestra capa de convolución de grafos ya funciona. 
Primero guardamos una matriz de features y una de adyacencia de ejemplo:

In [ ]:
feat_example = feat[1]
adj_example = DA[1]
print('Features:', feat_example.shape)
print('DA:', adj_example.shape)

Ahora inicializamos una `GraphConvolution`. Ten en cuenta que el tamaño de entrada de la primera capa 
es igual al tamaño de la matriz de features (25 tipos de átomos):

In [ ]:
conv = GraphConvolution(25, 100)
conv

Ahora podemos pasar el ejemplo por la convolución. Verás que la dimensión de los features habrá aumentado a 100:

In [ ]:
output = conv(feat_example, adj_example)
print('\nOutput:', output.size())

## Red Neuronal de Grafos

Para crear una red ahora, podemos usar nuevamente la clase `nn.Module`.

La semana pasada usamos `nn.Sequential` para combinar capas. 
Sin embargo, para las GNNs esto no es posible directamente, porque la convolución de grafos necesita **dos** entradas: 
la matriz de features y la matriz de adyacencia.

Por eso definimos la red como una clase que hereda de `nn.Module`:

- `__init__`: define las capas de la red
- `forward`: define cómo fluyen los datos por la red

Después de las capas de convolución, se aplica un **global mean pooling**: 
se promedia sobre todos los átomos para obtener una representación de la molécula completa.

In [ ]:
class GraphNN(nn.Module):
    def __init__(self):#in_features, out_features, size_labels):
        super().__init__()
        self.conv1 = GraphConvolution(25, 100)
        self.conv2 = GraphConvolution(100, 100)
        self.lin = nn.Linear(100, 1)
        
    def aggregate(self, convoluted_graph): # we use mean aggregation, max or min could also be used as hyperparameter
        return torch.mean(convoluted_graph, dim=0, keepdim=True)
        
    def forward(self, x, adj):
        x = self.conv1(x, adj)
        x = F.relu(x)
        x = self.conv2(x, adj)
        x = F.relu(x)
        x = self.aggregate(x)
        x = self.lin(x)
        return x

Ahora dividimos rápidamente el dataset en conjunto de entrenamiento y de prueba. 
Para esto simplemente usamos las primeras 1800 moléculas como entrenamiento y el resto como prueba:

In [ ]:
train_feat = feat[:1800]
train_DA = DA[:1800]
train_labels = labels[:1800]


test_feat = feat[1800:]
test_DA = DA[1800:]
test_labels = labels[1800:]

In [ ]:
gnn = GraphNN()
loss_function= nn.BCEWithLogitsLoss()
optimizer=optim.Adam(gnn.parameters(), lr =0.01)

In [ ]:
EPOCHS = 20

for i in range(EPOCHS):
    loss_list_train = []
    acc_list_train= []
    gnn.train()
    for k in range(len(train_feat)):
        optimizer.zero_grad()
    
        output=gnn(train_feat[k], train_DA[k]).flatten()

        loss=loss_function(output,train_labels[k])
        loss.backward()
        loss_list_train.append(loss.item())
        optimizer.step()

        acc_list_train.append(np.sum((torch.round(torch.sigmoid(output)) == train_labels[k]).detach().numpy()))
    loss_list_test = []
    acc_list_test= []
    gnn.eval()
    for k in range(len(test_feat)):
    
        output=gnn(test_feat[k], test_DA[k]).flatten()

        loss=loss_function(output,test_labels[k])
        loss_list_test.append(loss.item())
 

        acc_list_test.append(np.sum((torch.round(torch.sigmoid(output)) == test_labels[k]).detach().numpy()))
            
        
    print(i,"Train Loss: %.2f Train Accuracy: %.2f Test Loss: %.2f Test Accuracy: %.2f"
        % (np.mean(loss_list_train), np.mean(acc_list_train),np.mean(loss_list_test), np.mean(acc_list_test)))

Como puedes ver, el entrenamiento es lento y no muy exitoso. El modelo presentado aquí es muy, muy simple. 
En la práctica, las GNNs son mucho más complejas y requieren mucho más tiempo de entrenamiento.

Pero el principio fundamental es el mismo: pasar información entre nodos vecinos y aprender representaciones de moléculas completas. 
Las GNNs modernas como **Message Passing Neural Networks** (MPNN) o **Graph Attention Networks** (GAT) 
siguen este mismo principio pero con mecanismos más sofisticados.

Para uso práctico en drug discovery, se recomienda usar librerías especializadas como **PyTorch Geometric** o **DGL** 
que ofrecen implementaciones optimizadas y arquitecturas pre-entrenadas.

## Ejercicio:

Aquí puedes ver nuestra Red Neuronal de Grafos. 
El problema es que esta red no ofrece flexibilidad. 
Los pesos siempre se inicializan con el mismo tamaño. 
Queremos modificarla para que se pueda definir el tamaño de los features en cada capa al crear la red.

Para ello, agrega parámetros `__init__` que permitan especificar el número de features ocultos.

**¿Puedes modificar la clase `GraphNN` para que sea más flexible?**

In [ ]:
example_DA = test_DA[0]
example_feat = test_feat[0]

In [ ]:
class GraphNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GraphConvolution(25, 100)
        self.conv2 = GraphConvolution(100, 100)
        self.lin = nn.Linear(100, 1)
        
    def aggregate(self, convoluted_graph): 
        return torch.mean(convoluted_graph, dim=0, keepdim=True)
        
    def forward(self, x, adj):
        x = self.conv1(x, adj)
        x = F.relu(x)
        x = self.conv2(x, adj)
        x = F.relu(x)
        x = self.aggregate(x)
        x = self.lin(x)
        return x